In [ ]:
# [ignoring loop detection]
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import mnist

# 1. 數據準備 (MNIST 數字辨識)
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 標準化與重塑格式
x_train = x_train.reshape(60000, 28, 28, 1) / 255
x_test = x_test.reshape(10000, 28, 28, 1) / 255
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# 2. 建立我的創意神經網路
model = Sequential()

# --- 創意點 1：BatchNormalization (批次標準化) ---
# 就像給神經網路裝上穩定器，讓每一層的數據分佈更均勻，大幅提升訓練速度。
model.add(Conv2D(32, (3,3), padding='same', input_shape=(28,28,1), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))

# --- 創意點 2：Dropout (隨機失活) ---
# 隨機讓部分神經元休息，防止模型「死背」題目，這能顯著提升對新資料的辨識準確度。
model.add(Dropout(0.25))
model.add(Flatten())

# 全連接層 (增加神經元數量提升理解能力)
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

# 3. 組裝模型 (創意點 3：使用 Adam 優化器與分類專用損失函數)
model.compile(loss='categorical_crossentropy',
              optimizer=Adam(learning_rate=0.001),
              metrics=['accuracy'])

# 4. 開始訓練 (設定較大的 batch_size=256 以利用 GPU 進行超速訓練)
model.fit(x_train, y_train,
          batch_size=256,
          epochs=6,
          validation_data=(x_test, y_test))

# 5. 評估與預測結果
loss, acc = model.evaluate(x_test, y_test)
print(f'\n 測試正確率達標：{acc*100:.2f}%')

# 預測介面
def my_predict(n):
    y_predict = np.argmax(model.predict(x_test), axis=-1)
    print(f' 我的 CNN 預測結果是：{y_predict[n]}')
    plt.imshow(x_test[n].reshape(28,28), cmap='Greys')

# 測試第 100 號圖片
my_predict(100)
